# 06 – Chỉ mục FAISS


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

CHUNKS_PATH = PROJECT_ROOT / "knowledge" / "chunks" / "chunks.jsonl"
INDEX_DIR = PROJECT_ROOT / "knowledge" / "index"
INDEX_PATH = INDEX_DIR / "faiss.index"
META_PATH = INDEX_DIR / "chunks_meta.json"
INFO_PATH = INDEX_DIR / "build_info.json"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("chunks =", CHUNKS_PATH, "exists", CHUNKS_PATH.exists())

PROJECT_ROOT = d:\TTTN-VaxCare-2026\vaxcare-ai
chunks = d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl exists True


In [2]:
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print("n_chunks =", len(chunks))
texts = [(c.get("text") or "").strip() for c in chunks]
assert all(texts), "Có chunk text rỗng"
print("avg_len =", int(np.mean([len(t) for t in texts])))
print("sample keys", chunks[0].keys())

n_chunks = 100
avg_len = 625
sample keys dict_keys(['chunk_id', 'doc_id', 'title', 'text', 'source_file', 'tier', 'vaccine_ids', 'page'])


In [3]:
from app.chatbot.rag.embedder import load_embedder

# prefer_bge=True: thử BGE-M3; fail → TF-IDF fit trên corpus chunks
embedder = load_embedder(prefer_bge=True, corpus_for_tfidf=texts)
print("model =", embedder.model_name, "dim =", embedder.dim)

emb = embedder.encode(texts, batch_size=8)
print("embeddings shape", emb.shape, emb.dtype)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model = BAAI/bge-m3 dim = 1024


Inference Embeddings: 100%|██████████| 13/13 [03:24<00:00, 15.72s/it]

embeddings shape (100, 1024) float32


In [4]:
from app.chatbot.rag.vector_store import VectorStore

metas = []
for c in chunks:
    metas.append({
        "chunk_id": c.get("chunk_id"),
        "doc_id": c.get("doc_id"),
        "title": c.get("title"),
        "text": c.get("text"),
        "source_file": c.get("source_file"),
        "tier": c.get("tier"),
        "vaccine_ids": c.get("vaccine_ids") or [],
        "page": c.get("page"),
    })

store = VectorStore.build_from_embeddings(emb, metas, model_name=embedder.model_name)
print("index size", store.size, "dim", store.dim)

build_info = {
    "model": embedder.model_name,
    "dim": store.dim,
    "n_chunks": store.size,
    "built_at": datetime.now(timezone.utc).isoformat(),
    "chunks_path": str(CHUNKS_PATH),
}
store.save(INDEX_PATH, META_PATH, build_info=build_info)
print("saved", INDEX_PATH)
print("saved", META_PATH)
print("saved", INFO_PATH)

index size 100 dim 1024
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\faiss.index
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\chunks_meta.json
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\build_info.json


In [5]:
# Smoke test retrieval
queries = [
    "Khám sàng lọc trước tiêm chủng trẻ em",
    "Vaccine HPV phòng bệnh gì",
    "Chống chỉ định tiêm chủng",
]
for q in queries:
    qv = embedder.encode([q])[0]
    hits = store.search(qv, top_k=3, min_score=0.0)
    print("\nQ:", q)
    for h in hits:
        snippet = (h.get("text") or "")[:120].replace("\n", " ")
        print(f"  [{h['score']:.3f}] {h.get('doc_id','')[:40]} | {snippet}...")


Q: Khám sàng lọc trước tiêm chủng trẻ em
  [0.750] 6381586785791217441575-qđ-byt-về-việc-hư | khám sàng lọc tại bệnh viện. II. KHÁM SÀNG LỌC TRƯỚC TIÊM CHỦNG CHO TRẺ EM ĐỐI VỚI CƠ SỞ NGOÀI BỆNH VIỆN 1. Khám sàng lọ...
  [0.721] 6381586785791217441575-qđ-byt-về-việc-hư |  Văn phòng Chương trình Tiêm chủng mở rộng Quốc gia, Viện Vệ sinh dịch tễ Trung ương ThS. Lê Kim Dung Chuyên viên chính,...
  [0.718] 6381586785791217441575-qđ-byt-về-việc-hư | tuyến trên để được khám sàng lọc. + Tiếp nhận và tiêm chủng đối với các trường hợp đã được kiểm tra đánh giá tình trạng ...

Q: Vaccine HPV phòng bệnh gì
  [0.696] db-vaccine | Vaccine: Vắc xin HPV (Human Papillomavirus) Nhà sản xuất: MSD – Hoa Kỳ Phòng bệnh: Ung thư cổ tử cung, sùi mào gà và các...
  [0.569] db-protocol | Phác đồ: Phác đồ HPV 3 liều Vaccine: Vắc xin HPV (Human Papillomavirus) Mô tả: Phác đồ 3 liều cho nhóm tuổi lớn hơn...
  [0.566] db-protocol | Phác đồ: Phác đồ HPV 2 liều Vaccine: Vắc xin HPV (Human Papillomavirus) Mô tả: Phác